# 08 - Experiments & Analysis

This notebook demonstrates how to run systematic experiments with PINNs:
- Hyperparameter sweeps
- Ablation studies
- Loss landscape analysis
- Convergence analysis

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import matplotlib.pyplot as plt
import numpy as np
from pinns import Config, MLP, Trainer, set_seed
from pinns.equations.heat import HeatEquation1D

In [ ]:
set_seed(42)

# Experiment 1: Effect of network width
widths = [16, 32, 64, 128]
results_width = {}

for width in widths:
    print(f"\nTesting width={width}...")
    
    config = Config(
        name=f'exp_width_{width}',
        equation_name='heat_1d',
        equation_params={'alpha': 0.01},
        model={'type': 'mlp', 'hidden_layers': [width, width, width], 'activation': 'tanh'},
        training={'n_collocation': 500, 'n_boundary': 50, 'n_initial': 50, 'epochs': 1000, 'lr': 1e-3},
    )
    
    model = MLP(input_dim=2, output_dim=1, hidden_layers=[width, width, width], activation='tanh')
    eq = HeatEquation1D(alpha=0.01)
    
    trainer = Trainer(model=model, equation=eq, config=config, print_every=0)
    history = trainer.train()
    
    results_width[width] = history['total_loss'][-1]
    print(f"  Final loss: {history['total_loss'][-1]:.6e}")

plt.figure(figsize=(10, 5))
plt.bar(range(len(widths)), [results_width[w] for w in widths])
plt.xticks(range(len(widths)), [str(w) for w in widths])
plt.xlabel('Network Width')
plt.ylabel('Final Loss')
plt.yscale('log')
plt.title('Effect of Network Width on PINN Performance')
plt.show()

In [ ]:
# Experiment 2: Effect of collocation points
n_points_list = [100, 250, 500, 1000, 2000]
results_points = {}

for n_pts in n_points_list:
    print(f"\nTesting n_collocation={n_pts}...")
    
    config = Config(
        name=f'exp_pts_{n_pts}',
        equation_name='heat_1d',
        equation_params={'alpha': 0.01},
        model={'type': 'mlp', 'hidden_layers': [32, 32, 32], 'activation': 'tanh'},
        training={'n_collocation': n_pts, 'n_boundary': 50, 'n_initial': 50, 'epochs': 1000, 'lr': 1e-3},
    )
    
    model = MLP(input_dim=2, output_dim=1, hidden_layers=[32, 32, 32], activation='tanh')
    eq = HeatEquation1D(alpha=0.01)
    
    trainer = Trainer(model=model, equation=eq, config=config, print_every=0)
    history = trainer.train()
    
    results_points[n_pts] = history['total_loss'][-1]
    print(f"  Final loss: {history['total_loss'][-1]:.6e}")

plt.figure(figsize=(10, 5))
plt.plot(n_points_list, [results_points[n] for n in n_points_list], 'bo-')
plt.xlabel('Number of Collocation Points')
plt.ylabel('Final Loss')
plt.yscale('log')
plt.title('Effect of Collocation Points on PINN Performance')
plt.show()

In [ ]:
# Experiment 3: Loss landscape visualization
set_seed(42)

config = Config(
    name='loss_landscape',
    equation_name='heat_1d',
    equation_params={'alpha': 0.01},
    model={'type': 'mlp', 'hidden_layers': [32, 32], 'activation': 'tanh'},
    training={'n_collocation': 200, 'n_boundary': 20, 'n_initial': 20, 'epochs': 500, 'lr': 1e-3},
)

model = MLP(input_dim=2, output_dim=1, hidden_layers=[32, 32], activation='tanh')
eq = HeatEquation1D(alpha=0.01)

trainer = Trainer(model=model, equation=eq, config=config, print_every=0)
history = trainer.train()

# Plot loss trajectory
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history['total_loss'], label='Total')
plt.plot(history['physics_loss'], label='Physics')
plt.plot(history['boundary_loss'], label='Boundary')
plt.plot(history['initial_loss'], label='Initial')
plt.yscale('log')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss Components Over Time')

plt.subplot(1, 2, 2)
losses = np.array(history['total_loss'])
window = 10
smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
plt.plot(smoothed)
plt.xlabel('Iteration')
plt.ylabel('Smoothed Loss')
plt.yscale('log')
plt.title('Convergence Trend')

plt.tight_layout()
plt.show()

In [ ]:
print("Experiment Summary:")
print("="*50)
print("\n1. Width experiments show that larger networks")
print("   generally achieve lower final losses.")
print("\n2. More collocation points improve accuracy")
print("   but with diminishing returns.")
print("\n3. PINNs can struggle with multi-scale problems")
print("   - consider adaptive sampling or feature engineering.")